For this to work:

open "Teilnetz_Invest_Resultate_V24_1300MCHF_Chunk200.xslx"
and export "Entwicklung Teilnetz" as .csv
and rename it "data1.csv"
press run :)

In [59]:
import pandas as pd

# Import data.csv with semicolon delimiter and correct encoding
df = pd.read_csv('data1.csv', sep=';', encoding='latin-1')


C:\Users\z184922\AppData\Local\Temp\ipykernel_19612\2327922025.py:4: DtypeWarning: Columns (0: EOL, 1: Erneuerung_laufendes_Jahr, 2: Erneuerungskosten, 3: Remaining_LC) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data1.csv', sep=';', encoding='latin-1')


In [60]:
df.head(5)


,Streckenkategorie Level 1,Streckenkategorie,Streckenkategorie Level 2,Linie,Linie Level 2,Linie Level 3,Berichtsjahr,Gattungsname,Anlagentypname,ER_1,...,EXIT,Betrachtungsjahr,Erneuerungskosten_Year,Threshold,WBW_Anteil,Substanz_NachErneuerung,Remaining_LC,WbWxSqG,Processing_Order,Kostenstellen
0,Regionalstrecke,0,200250 - Le Day - (Vallorbe),200250,200 - Renens VD Ouest - Vallorbe,200250 - Le Day - (Vallorbe),2025,SAZ - Sicherungsanlagen und Zugbeeinflussung,Weichenheizungen,"0,011",...,FALSCH,2025,0,"3,8",100,1,0,4725000,5,200250
1,Regionalstrecke,0,200300 - Vallorbe (communauté),200300,203 - Frasne - Vallorbe,200300 - Vallorbe (communauté),2025,NS - Niederspannungs- und Telekommunikationsan...,NSV_Kundeninformation,"0,0045",...,FALSCH,2025,0,5,100,1,0,500000,5,200300
2,Hauptstrecke,0,230210 - Laufen - (Aesch),230210,230 - Delémont Est - Basel SBB Ost,230210 - Laufen - (Aesch),2025,NS - Niederspannungs- und Telekommunikationsan...,NSV_Kundeninformation,"0,0045",...,FALSCH,2025,0,"4,7",100,1,0,40000,5,230210
3,Hauptstrecke,0,226200 - Choindez - Delémont,226200,226 - Sonceboz-Sombeval - Moutier - Delémont,226200 - Choindez - Delémont,2025,NS - Niederspannungs- und Telekommunikationsan...,NSV_Kundeninformation,"0,0045",...,FALSCH,2025,0,"4,7",100,1,0,41000,5,226200
4,Regionalstrecke,0,200200 - (Daillens) - (Le Day),200200,200 - Renens VD Ouest - Vallorbe,200200 - (Daillens) - (Le Day),2025,NS - Niederspannungs- und Telekommunikationsan...,NSV_Kundeninformation,"0,0045",...,FALSCH,2025,0,5,100,1,0,95000,5,200200


In [61]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np


# Display basic info about the data
print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Available years:", sorted(df['Betrachtungsjahr'].unique()))
print("Sample Linie Level 3 values:", df['Linie Level 3'].unique()[:5])

Data shape: (1048575, 54)
Columns: ['Streckenkategorie Level 1', 'Streckenkategorie', 'Streckenkategorie Level 2', 'Linie', 'Linie Level 2', 'Linie Level 3', 'Berichtsjahr', 'Gattungsname', 'Anlagentypname', 'ER_1', 'ER_2', 'ER_3', 'ER_4', 'ER_5', 'LC100', 'LC75', 'LC50', 'LC25', 'EOL', 'ExternalId', 'Anlagenalter (Jahre)', 'Wiederbeschaffungswert (CHF)', 'Effektive SOLL-Gesamtlebensdauer (Jahre)', 'Effektive SOLL-Restlebensdauer absolut (Jahre)', 'Substanz - Quality Aspect Grade', 'Zustandsklasse Substanz - Quality Aspect Grade', 'Anlagenbezeichnung', 'Reference_Anlage', 'Y_EOL', 'GUID', 'Strategy', 'Gradient1Tier', 'Gradient2Tier', 'Gradient3Tier', 'Gradient4Tier', 'RGradient', 'StrategyType', 'Erneuerung_laufendes_Jahr', 'Unterhalt_laufendes_Jahr', 'Substanzerhalt_laufendes_Jahr', 'Erneuerungskosten', 'WBWxZK', 'WBWK', 'Flag_Updated', 'EXIT', 'Betrachtungsjahr', 'Erneuerungskosten_Year', 'Threshold', 'WBW_Anteil', 'Substanz_NachErneuerung', 'Remaining_LC', 'WbWxSqG', 'Processing_Ord

In [65]:
# Prepare data for visualization
# Group by Linie Level 3, Quality Grade Category, and Year, counting unique GUID

# Convert Quality Aspect Grade to numeric (handling any non-numeric values)
df['Quality_Grade_Numeric'] = pd.to_numeric(
    df['Substanz - Quality Aspect Grade'].astype(str).str.replace(',', '.'),
    errors='coerce'
)

# Create quality grade category (< 4 or > 4)
df['Quality_Category'] = df['Quality_Grade_Numeric'].apply(
    lambda x: 'Grade < 4' if pd.notna(x) and x < 4 else ('Grade > 4' if pd.notna(x) else 'No Grade')
)

# Get available years
years = sorted(df['Betrachtungsjahr'].dropna().unique())
print(f"Years available: {years}")

# Function to get data for a specific year
def get_year_data(year):
    df_year = df[df['Betrachtungsjahr'] == year].copy()
    
    # Count unique GUID per Linie Level 3 and Quality Category
    grouped = df_year.groupby(['Linie Level 3', 'Quality_Category'])['GUID'].nunique().reset_index()
    grouped.columns = ['Linie Level 3', 'Quality_Category', 'Count']
    
    return grouped

# Test with first year
test_data = get_year_data(years[0])
print(f"\nSample data for year {int(years[0])}:")
print(test_data.head(90))

Years available: [np.int64(2025), np.int64(2026), np.int64(2027), np.int64(2028), np.int64(2029), np.int64(2030), np.int64(2031), np.int64(2032), np.int64(2033), np.int64(2034), np.int64(2035), np.int64(2036), np.int64(2037), np.int64(2038), np.int64(2039), np.int64(2040), np.int64(2041), np.int64(2042), np.int64(2043), np.int64(2044), np.int64(2045), np.int64(2046), np.int64(2047), np.int64(2048), np.int64(2049)]

Sample data for year 2025:
                                   Linie Level 3 Quality_Category  Count
0                 200100 - (Renens) - (Bussigny)        Grade < 4    136
1             200150 - Bussigny - Daillens [bif]        Grade < 4   2119
2             200150 - Bussigny - Daillens [bif]        Grade > 4   1596
3                 200200 - (Daillens) - (Le Day)        Grade < 4   1441
4                 200200 - (Daillens) - (Le Day)        Grade > 4   2626
5                   200250 - Le Day - (Vallorbe)        Grade < 4   1015
6                   200250 - Le Day - (Vall

In [67]:
import plotly.graph_objects as go

# 1. Get ALL unique Linie Level 3 names across all years combined
all_linies = set()
for year in years:
    df_year = get_year_data_filtered(year, unique_anlagentypnames)
    all_linies.update(df_year['Linie Level 3'].dropna().unique())

# Sort them alphabetically
sorted_linies = sorted(list(all_linies))

# Prepare all frames for the slider
frames = []

for year in years:
    df_year = get_year_data_filtered(year, selected_anlagentypnames)
    
    # Pivot data to have Grade < 4 and Grade > 4 as separate traces
    grade_lt4 = df_year[df_year['Quality_Category'] == 'Grade < 4'].set_index('Linie Level 3')['Count']
    grade_gt4 = df_year[df_year['Quality_Category'] == 'Grade > 4'].set_index('Linie Level 3')['Count']
    
    # Reindex with the global sorted lines (adds missing lines with 0 count)
    grade_lt4 = grade_lt4.reindex(sorted_linies, fill_value=0)
    grade_gt4 = grade_gt4.reindex(sorted_linies, fill_value=0)
    
    # Create frame for this year
    frame = go.Frame(
        data=[
            go.Bar(
                x=grade_lt4.values,
                y=grade_lt4.index,
                name='Quality < 4',
                orientation='h',
                marker=dict(color='#006E09'),
                hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
            ),
            go.Bar(
                x=grade_gt4.values,
                y=grade_gt4.index,
                name='Quality > 4',
                orientation='h',
                marker=dict(color='#FF0000'),
                hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
            )
        ],
        name=str(int(year))
    )
    frames.append(frame)


sorted_linies = sorted(df['Linie Level 3'].dropna().unique())

# Create initial figure using the first year's data, reindexed to the global list
initial_data = get_year_data_filtered(years[0], selected_anlagentypnames)
grade_lt4_initial = initial_data[initial_data['Quality_Category'] == 'Grade < 4'].set_index('Linie Level 3')['Count']
grade_gt4_initial = initial_data[initial_data['Quality_Category'] == 'Grade > 4'].set_index('Linie Level 3')['Count']
grade_lt4_initial = grade_lt4_initial.reindex(sorted_linies, fill_value=0)
grade_gt4_initial = grade_gt4_initial.reindex(sorted_linies, fill_value=0)
fig = go.Figure(
    data=[
        go.Bar(
            x=grade_lt4_initial.values,
            y=grade_lt4_initial.index,
            name='Quality < 4',
            orientation='h',
            marker=dict(color="#006E09"),
            hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
        ),
        go.Bar(
            x=grade_gt4_initial.values,
            y=grade_gt4_initial.index,
            name='Quality > 4',
            orientation='h',
            marker=dict(color="#FF0000"),
            hovertemplate='%{y}<br>Count: %{x}<extra></extra>'
        )
    ],
    frames=frames
)

# Add year slider
sliders = [
    {
        'active': 0,
        'yanchor': 'top',
        'y': 0,
        'xanchor': 'left',
        'x': 0.1,
        'currentvalue': {
            'prefix': 'Year: ',
            'visible': True,
            'xanchor': 'center',
            'font': {'size': 16}
        },
        'transition': {'duration': 300},
        'pad': {'b': 10, 't': 50},
        'len': 0.9,
        'steps': [
            {
                'args': [
                    [str(int(year))],
                    {
                        'frame': {'duration': 300, 'redraw': True},
                        'mode': 'immediate',
                        'transition': {'duration': 300}
                    }
                ],
                'method': 'animate',
                'label': str(int(year))
            }
            for year in years
        ]
    }
]

fig.update_layout(
    sliders=sliders,
    title='Unique Assets by Railway Line (Linie Level 3) - Quality Grade Comparison',
    xaxis_title='Count of Unique Assets ',
    yaxis_title='Linie Level 3',
    
    # ---> INSERT THIS YAXIS DICTIONARY HERE <---
    yaxis=dict(
        type='category',
        categoryorder='array',
        categoryarray=sorted_linies
    ),
    # -------------------------------------------
    
    barmode='stack',
    height=max(900, len(sorted_linies) * 20),
    hovermode='closest',
    showlegend=True,
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top')
)

fig.show()
